In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.metrics import mean_squared_error # Squared will be false for scoring
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline



plt.style.use('ggplot')
pd.set_option('display.max_columns', 200)

# Reading the CSV Files


## Training Dataset

First I am going to read the **train dataset**. 

In [ ]:
train_data = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
print(f"Training Dataset Shape: {train_data.shape}")


I am going to use the `.head()` method to see what type of data we are using the **training dataset**. 

In [ ]:
train_data.head()

Now I am going to use the `.info()` method in order to see what **dtypes** we are dealing with. 

In [ ]:
train_data.info()

Let's see a visualization of **SalePrice** distribution.

In [ ]:
sns.histplot(train_data["SalePrice"], bins=100, kde=True, color='Blue')
plt.title("Distribution of SalePrice")
plt.tight_layout()
plt.show

## Test Dataset

We are now going to do the same thing for the **test dataset**. 

In [ ]:
test_data = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')
print(f"Testing Dataset Shape: {test_data.shape}")

In [ ]:
test_data.head()

In [ ]:
test_data.info()

# Cleaning the Datasets

## Getting the numerics

 ### Training Data

Now I am going to get all the numerics in the datasets

In [ ]:
numerics_train = ['int64', 'float64']
numeric_train = train_data.select_dtypes(include=numerics_train).copy()
print(numeric_train.shape)
numeric_train.head()

### Testing Data

In [ ]:
numerics_test = ['int64', 'float64']
numeric_test = test_data.select_dtypes(include=numerics_test).copy()
print(numeric_test.shape)
numeric_test.head()

## One-Hot Encoding

### Train Data

I feel as if it is pretty important that we include the type of neighborhood for our model. I will perform One-Hot encoding and I will combine it with our numeric dataset for our training data. 

In [ ]:
neighborhood = train_data['Neighborhood']
neighborhood = pd.get_dummies(neighborhood).astype(int)
print(neighborhood.shape)
print(neighborhood.head())

numeric_train = pd.concat([numeric_train, neighborhood], axis=1)
print(numeric_train.shape)
numeric_train.head()


### Test Data

In [ ]:
neighborhood = test_data['Neighborhood']
neighborhood = pd.get_dummies(neighborhood).astype(int)
print(neighborhood.shape)
print(neighborhood.head())

numeric_test = pd.concat([numeric_test, neighborhood], axis=1)
print(numeric_test.shape)
numeric_test.head()

## Removing Columns and Rows with missing data

### Train Data

First we are going to remove any columns that have over **50 null entries**.

In [ ]:
missing_values_count = numeric_train.isnull().sum()
columns_to_drop = missing_values_count[missing_values_count > 50].index
initial_columns = numeric_train.shape[1]
numeric_train = numeric_train.drop(columns=columns_to_drop, axis=1)
final_columns = numeric_train.shape[1]
print(f"Columns dropped: {initial_columns - final_columns}")
print(f"New shape: {numeric_train.shape}")

Now we are going to remove **any rows** that have **null entries**.

In [ ]:
nrows_before = numeric_train.shape[0]
numeric_train = numeric_train.dropna()
nrows_after = numeric_train.shape[0]
print(f"** Removed {nrows_before - nrows_after} rows **")
print(numeric_train.shape)
numeric_train.describe()

### Test Data

In [ ]:
missing_values_count = numeric_test.isnull().sum()
columns_to_drop = missing_values_count[missing_values_count > 50].index
initial_columns = numeric_test.shape[1]
numeric_test = numeric_test.drop(columns=columns_to_drop, axis=1)
final_columns = numeric_test.shape[1]
print(f"Columns dropped: {initial_columns - final_columns}")
print(f"New shape: {numeric_test.shape}")
numeric_test.describe()

# Visualizations

## Correlation 

In [ ]:
X = numeric_train.drop(columns=['Id','SalePrice']).copy()
y = numeric_train['SalePrice'].copy()

In [ ]:
correlations = X.corrwith(y).abs().sort_values(ascending=False)
X = X[correlations.index]
X.head()

In [ ]:
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
sns.histplot(numeric_train['OverallQual'], bins = 50, kde=True, ax=axes[0, 0]).set_title('Distribution of OverallQual')
sns.histplot(numeric_train['GrLivArea'], bins = 50, kde=True, ax=axes[0, 1]).set_title('Distribution of GrLivArea')
sns.histplot(numeric_train['GarageCars'], bins = 50, kde=True, ax=axes[1, 0]).set_title('Distribution of GarageCars')
sns.histplot(numeric_train['GarageArea'], bins = 50, kde=True, ax=axes[1, 1]).set_title('Distribution of Garage Area')
sns.histplot(numeric_train['TotalBsmtSF'], bins = 50, kde=True, ax=axes[2, 0]).set_title('Distribution of TotalBsmtSF')
sns.histplot(numeric_train['1stFlrSF'], bins = 50, kde=True, ax=axes[2, 1]).set_title('Distribution of 1stFlrSF')

plt.tight_layout()
plt.show()

# Predicting


In [ ]:
# 1) Split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ------------------------------
# 2) Base pipeline
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression()),
])

# ------------------------------
# 3) Param grids (HGBR replaces classic GBoost)
param_grids = [
    {
        "regressor": [LinearRegression()],
        "scaler": ["passthrough", StandardScaler()],
    },
    {
        "regressor": [DecisionTreeRegressor(random_state=42)],
        "regressor__max_depth": [None, 10, 20],
        "regressor__min_samples_split": [2, 5, 10],
        "scaler": ["passthrough"],   # no scaling for trees
    },
    {
        "regressor": [RandomForestRegressor(random_state=42, n_jobs=-1)],
        "regressor__n_estimators": [200, 500],
        "regressor__max_depth": [None, 10, 20],
        "scaler": ["passthrough"],   # no scaling for RF
    },
    {
        "regressor": [HistGradientBoostingRegressor(random_state=42,
                                                    early_stopping=False)],
        # HGBR hyperparams (note: uses max_iter, not n_estimators)
        "regressor__max_iter": [300, 600],
        "regressor__learning_rate": [0.05, 0.1],
        "regressor__max_depth": [None, 6, 10],
        "regressor__min_samples_leaf": [10, 20],
        "regressor__max_bins": [255, 128],
        "regressor__l2_regularization": [0.0, 1.0],
        "scaler": ["passthrough"],   # never scale for HGBR
    },
]

# ------------------------------
# 4) Grid Search Across Models
cv = KFold(n_splits=5, shuffle=True, random_state=42)
best_models = []

for grid in param_grids:
    gs = GridSearchCV(
        estimator=pipe,
        param_grid=grid,
        scoring="neg_root_mean_squared_error",  # RMSE (negated)
        cv=cv,
        n_jobs=-1,
        verbose=0,
        refit=True,
    )
    gs.fit(X_train, y_train)
    best_models.append(gs)

    print(f"Model: {grid['regressor'][0].__class__.__name__}")
    print("Best Params:", gs.best_params_)
    print("CV RMSE:", -gs.best_score_)
    print("-" * 40)

# ------------------------------
# 5) Pick the Overall Best Model
best_overall = max(best_models, key=lambda x: x.best_score_)  # less neg = better
final_model = best_overall.best_estimator_

print("BEST MODEL SELECTED:", final_model["regressor"].__class__.__name__)
print("Best Parameters:", best_overall.best_params_)

# Validation RMSE
y_pred = final_model.predict(X_valid)
val_rmse = mean_squared_error(y_valid, y_pred, squared=False)
print("Validation RMSE:", val_rmse)

In [ ]:
# 1. Drop 'Id' or any extra columns from the test set
numeric_test = numeric_test.drop(columns=['Id'], errors='ignore')

# 2. Align test columns with training columns in the same order
numeric_test = numeric_test[X_train.columns]

# 3. Make predictions
predictions = final_model.predict(numeric_test)

# 4. Create submission file
submission = pd.DataFrame({
    "Id": test_data["Id"],      # Use original 'test_data' for Id column
    "SalePrice": predictions
})

submission.to_csv("submission.csv", index=False)
print(" submission.csv created successfully!")
